In [ ]:
# 1. Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split

# 2. Load the dataset
df = pd.read_csv('triage_1000_records.csv')

# 3. Explore the dataset
print("Columns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 Rows:\n", df.head())
print("\nMissing Values:\n", df.isnull().sum())

# Summary statistics for numerical columns
print("Summary statistics (numerical):")
display(df.describe())

# Summary statistics for categorical columns
print("\nSummary statistics (categorical):")
display(df.describe(include=['object']))

# Visualize distributions for numerical features
import matplotlib.pyplot as plt
import seaborn as sns

numerical_cols = df.select_dtypes(include=['number']).columns
for col in numerical_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

# Visualize distributions for categorical features
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    plt.figure(figsize=(6, 4))
    df[col].value_counts().plot(kind='bar')
    plt.title(f'Value counts of {col}')
    plt.ylabel('Count')
    plt.show()

# Correlation heatmap for numerical features (fix: select only numeric columns)
numeric_df = df.select_dtypes(include=['number'])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Check for outliers using boxplots
for col in numerical_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

# 4. Summarize target variable (replace 'target_column' with actual target column name)
# print("\nTarget Variable Distribution:\n", df['target_column'].value_counts())

# 5. Plan the data split for three clinics
clinic1, temp = train_test_split(df, test_size=2/3, random_state=42)
clinic2, clinic3 = train_test_split(temp, test_size=0.5, random_state=42)

print(f"\nClinic 1 records: {len(clinic1)}")
print(f"Clinic 2 records: {len(clinic2)}")
print(f"Clinic 3 records: {len(clinic3)}")

In [ ]:
# Save the three clinic datasets as separate CSV files
clinic1.to_csv('clinic1_data.csv', index=False)
clinic2.to_csv('clinic2_data.csv', index=False)
clinic3.to_csv('clinic3_data.csv', index=False)

print("Clinic datasets saved as clinic1_data.csv, clinic2_data.csv, and clinic3_data.csv")

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
import joblib

# 2. Load datasets
clinic1 = pd.read_csv('clinic1_data.csv')
clinic2 = pd.read_csv('clinic2_data.csv')
clinic3 = pd.read_csv('clinic3_data.csv')

# 3. Select only the most correlated features and target
features = [
    'age',
    'vital_signs_oxygen_saturation',
    'vital_signs_diastolic_bp',
    'vital_signs_systolic_bp'
]
target = 'triage_level'

# 4. Preprocess: Handle missing values
def preprocess(df):
    df = df.copy()
    # Impute missing values for features
    imputer = SimpleImputer(strategy='mean')
    df[features] = imputer.fit_transform(df[features])
    # Encode target as integer if not already
    df[target] = pd.to_numeric(df[target], errors='coerce')
    df = df.dropna(subset=[target])
    df[target] = df[target].astype(int)
    return df

clinic1 = preprocess(clinic1)
clinic2 = preprocess(clinic2)
clinic3 = preprocess(clinic3)

# 5. Split train/test for each clinic
def split_data(df):
    return train_test_split(df[features], df[target], test_size=0.3, random_state=42, stratify=df[target])

X1_train, X1_test, y1_train, y1_test = split_data(clinic1)
X2_train, X2_test, y2_train, y2_test = split_data(clinic2)
X3_train, X3_test, y3_train, y3_test = split_data(clinic3)

# 6. Train three different models
model1 = LogisticRegression(max_iter=1000).fit(X1_train, y1_train)
model2 = DecisionTreeClassifier().fit(X2_train, y2_train)
model3 = RandomForestClassifier(n_estimators=100).fit(X3_train, y3_train)

# Save individual models
joblib.dump(model1, 'clinic1_model.joblib')
joblib.dump(model2, 'clinic2_model.joblib')
joblib.dump(model3, 'clinic3_model.joblib')

# 7. Evaluate each model
print("=== Clinic 1 Model (Logistic Regression) ===")
y1_pred = model1.predict(X1_test)
print(classification_report(y1_test, y1_pred))

print("=== Clinic 2 Model (Decision Tree) ===")
y2_pred = model2.predict(X2_test)
print(classification_report(y2_test, y2_pred))

print("=== Clinic 3 Model (Random Forest) ===")
y3_pred = model3.predict(X3_test)
print(classification_report(y3_test, y3_pred))

# 8. Federated learning: average predictions (soft voting) on combined test set
test_X = pd.concat([X1_test, X2_test, X3_test])
test_y = pd.concat([y1_test, y2_test, y3_test])

pred1 = model1.predict_proba(test_X)
pred2 = model2.predict_proba(test_X)
pred3 = model3.predict_proba(test_X)

# Ensure all models output the same number/order of classes
classes = np.unique(np.concatenate([model1.classes_, model2.classes_, model3.classes_]))
def align_proba(pred, model, classes):
    proba = np.zeros((pred.shape[0], len(classes)))
    for idx, c in enumerate(model.classes_):
        proba[:, np.where(classes == c)[0][0]] = pred[:, idx]
    return proba

pred1 = align_proba(pred1, model1, classes)
pred2 = align_proba(pred2, model2, classes)
pred3 = align_proba(pred3, model3, classes)

# Federated (ensemble) prediction: average probabilities
federated_pred_proba = (pred1 + pred2 + pred3) / 3
federated_pred = np.argmax(federated_pred_proba, axis=1)
federated_pred_labels = classes[federated_pred]

print("=== Federated (Ensemble) Model ===")
print(classification_report(test_y, federated_pred_labels))

# 9. Meta-model: Use outputs of three models as features
meta_features = np.hstack([pred1, pred2, pred3])
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(meta_features, test_y)

# Save meta-model
joblib.dump(meta_model, 'federated_meta_model.joblib')

# Evaluate meta-model
meta_pred = meta_model.predict(meta_features)
print("=== Meta-model (Stacked) ===")
print(classification_report(test_y, meta_pred))

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# ==============================
# 1. Configuration
# ==============================
FEATURES = [
    "age",
    "vital_signs_oxygen_saturation",
    "vital_signs_diastolic_bp",
    "vital_signs_systolic_bp"
]
TARGET = "triage_level"

CLIENT_FILES = [
    "clinic1_data.csv",
    "clinic2_data.csv",
    "clinic3_data.csv"
]

TEST_SIZE = 0.30
RANDOM_STATE = 42

# Federated hyperparameters
ROUNDS = 50
LOCAL_EPOCHS = 30
LEARNING_RATE = 0.01

# ==============================
# 2. Data preprocessing
# ==============================
def preprocess_client(df, features, target):
    df = df.copy()
    # Convert all features to numeric, coercing bad values
    for col in features:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    # Local mean imputation
    local_means = df[features].mean(numeric_only=True)
    df[features] = df[features].fillna(local_means)
    # Clean target
    df[target] = pd.to_numeric(df[target], errors="coerce")
    df = df.dropna(subset=[target])
    df[target] = df[target].astype(int)
    return df

def load_clients(client_files):
    clients = []
    for f in client_files:
        c = pd.read_csv(f)
        c = preprocess_client(c, FEATURES, TARGET)
        clients.append(c)
    return clients

clients_df = load_clients(CLIENT_FILES)

# Split each client locally
client_splits = []
for i, cdf in enumerate(clients_df, start=1):
    X = cdf[FEATURES].values.astype(np.float64)
    y = cdf[TARGET].values.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    client_splits.append((X_train, X_test, y_train, y_test))
    print(f"Client {i}: train={len(y_train)}, test={len(y_test)}")

# Build global class mapping
all_train_labels = np.concatenate([s[2] for s in client_splits])
global_classes = np.sort(np.unique(all_train_labels))
class_to_index = {c: i for i, c in enumerate(global_classes)}
index_to_class = {i: c for c, i in class_to_index.items()}

def encode_labels(y):
    return np.array([class_to_index[v] for v in y], dtype=np.int64)

# ==============================
# 3. Federated model (Softmax Regression)
# ==============================
def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def one_hot(y_idx, num_classes):
    m = y_idx.shape[0]
    y_oh = np.zeros((m, num_classes), dtype=np.float64)
    y_oh[np.arange(m), y_idx] = 1.0
    return y_oh

def local_train_fedavg(W_global, b_global, X, y_idx, lr=0.05, local_epochs=5):
    # Start from current global weights
    W = W_global.copy()
    b = b_global.copy()
    n = X.shape[0]
    C = b.shape[0]
    y_oh = one_hot(y_idx, C)
    for _ in range(local_epochs):
        logits = X @ W + b
        probs = softmax(logits)
        grad_logits = (probs - y_oh) / n
        grad_W = X.T @ grad_logits
        grad_b = np.sum(grad_logits, axis=0)
        W -= lr * grad_W
        b -= lr * grad_b
    return W, b, n

def fedavg_round(client_models):
    # client_models: list of (W, b, n_samples)
    total_n = sum(n for _, _, n in client_models)
    W_avg = np.zeros_like(client_models[0][0])
    b_avg = np.zeros_like(client_models[0][1])
    for W, b, n in client_models:
        weight = n / total_n
        W_avg += weight * W
        b_avg += weight * b
    return W_avg, b_avg

def predict_with_weights(X, W, b):
    probs = softmax(X @ W + b)
    pred_idx = np.argmax(probs, axis=1)
    pred_labels = np.array([index_to_class[i] for i in pred_idx])
    return pred_labels, probs

# Initialize global model
num_features = len(FEATURES)
num_classes = len(global_classes)
W_global = np.zeros((num_features, num_classes), dtype=np.float64)
b_global = np.zeros((num_classes,), dtype=np.float64)

# ==============================
# 4. Federated training loop
# ==============================
for rnd in range(1, ROUNDS + 1):
    local_updates = []
    for X_train, X_test, y_train, y_test in client_splits:
        y_train_idx = encode_labels(y_train)
        W_local, b_local, n_local = local_train_fedavg(
            W_global,
            b_global,
            X_train,
            y_train_idx,
            lr=LEARNING_RATE,
            local_epochs=LOCAL_EPOCHS
        )
        local_updates.append((W_local, b_local, n_local))
    # Server aggregates only the shared weights
    W_global, b_global = fedavg_round(local_updates)
    # Optional monitoring on combined test set
    X_test_all = np.vstack([s[1] for s in client_splits])
    y_test_all = np.concatenate([s[3] for s in client_splits])
    y_pred_all, _ = predict_with_weights(X_test_all, W_global, b_global)
    acc = accuracy_score(y_test_all, y_pred_all)
    if rnd == 1 or rnd % 5 == 0 or rnd == ROUNDS:
        print(f"Round {rnd:02d} | Global test accuracy: {acc:.4f}")

# ==============================
# 5. Final evaluation
# ==============================
X_test_all = np.vstack([s[1] for s in client_splits])
y_test_all = np.concatenate([s[3] for s in client_splits])
y_pred_all, y_prob_all = predict_with_weights(X_test_all, W_global, b_global)

print("\n=== Federated Global Model (FedAvg, Weight Sharing) ===")
print("Accuracy:", accuracy_score(y_test_all, y_pred_all))
print(classification_report(y_test_all, y_pred_all))

# ==============================
# 6. Save global federated model
# ==============================
federated_model = {
    "weights": W_global,
    "bias": b_global,
    "features": FEATURES,
    "global_classes": global_classes,
    "class_to_index": class_to_index,
    "index_to_class": index_to_class,
    "rounds": ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "learning_rate": LEARNING_RATE
}

joblib.dump(federated_model, "federated_global_model_fedavg.joblib")
print("Saved: federated_global_model_fedavg.joblib")

Client 1: train=233, test=100
Client 2: train=233, test=100
Client 3: train=233, test=101
Round 01 | Global test accuracy: 0.3289
Round 05 | Global test accuracy: 0.4385
Round 10 | Global test accuracy: 0.4452
Round 15 | Global test accuracy: 0.3389
Round 20 | Global test accuracy: 0.3223
Round 25 | Global test accuracy: 0.4684
Round 30 | Global test accuracy: 0.3887
Round 35 | Global test accuracy: 0.4086
Round 40 | Global test accuracy: 0.4153
Round 45 | Global test accuracy: 0.4153
Round 50 | Global test accuracy: 0.4751

=== Federated Global Model (FedAvg, Weight Sharing) ===
Accuracy: 0.4750830564784053
              precision    recall  f1-score   support

           1       0.73      0.78      0.75        51
           2       0.77      0.27      0.40        84
           3       0.38      0.96      0.54        83
           4       0.00      0.00      0.00        60
           5       0.00      0.00      0.00        23

    accuracy                           0.48       301
   m

c:\Users\Surya_prakash\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Surya_prakash\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Surya_prakash\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.